In [10]:
from pathlib import Path
import csv
import json

BASE = Path("/Users/amandaeames/Documents/gitrepo/CityCatalyst-global-data/dataset-review/reviews/cl-ssg/cl-ssg-finance/releases/v1/data")

FUNDING_INPUT = BASE / "funding_database_v1_sin_duplicados.csv"
ACTION_INPUT = BASE / "action_funding_matching_inputs.csv"

FUNDING_NORMALIZED_OUT = BASE / "funding_database_v1_normalized.csv"
ACTION_FUNDABILITY_OUT = BASE / "action_fundability_scores_v1.csv"

In [11]:
# 1) Normalize funding file

def norm_text(v):
    v = (v or "").strip()
    return "" if (not v or v.lower() == "no especificado") else v

def norm_lower(v):
    v = norm_text(v)
    return v.lower() if v else ""

def split_tokens(v):
    v = (v or "").strip()
    if not v:
        return []
    return [t.strip() for t in v.split(",") if t.strip() and t.strip().lower() != "no especificado"]

def dedup_sorted(values):
    return sorted(set(values), key=lambda x: x.lower())

with FUNDING_INPUT.open("r", encoding="utf-8-sig", newline="") as f:
    funding_rows = list(csv.DictReader(f))

norm_fields = [
    "row_id", "opportunity_id", "opportunity_name", "source_actor_name",
    "source_actor_type", "instrument_type", "geographic_scope", "sector_scope",
    "status", "application_window", "source_url", "provider_actor_id",
    "notes_internal", "last_updated_at", "doc_source", "duplicate_group_id",
    "duplicate_decision", "eligible_actor_types_array", "gpc_sector_array",
    "gpc_subsector_array", "gpc_sector_raw"
]

with FUNDING_NORMALIZED_OUT.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=norm_fields)
    w.writeheader()

    for r in funding_rows:
        raw_row_id = (r.get("id") or r.get("n°_id") or "").strip()
        try:
            row_id = str(int(raw_row_id)) if raw_row_id else ""
        except ValueError:
            row_id = ""

        gpc_raw = (r.get("gpc_sector") or "").strip()
        gpc_tokens = split_tokens(gpc_raw)
        gpc_sector = []
        gpc_subsector = []
        for tok in gpc_tokens:
            if "–" in tok:
                gpc_sector.append(tok.split("–", 1)[0].strip())
                gpc_subsector.append(tok)
            else:
                gpc_sector.append(tok)

        eligible = dedup_sorted([t.lower() for t in split_tokens(r.get("eligible_actor_types"))])
        gpc_sector = dedup_sorted([x for x in gpc_sector if x])
        gpc_subsector = dedup_sorted([x for x in gpc_subsector if x])

        decision = (r.get("Desición ") or "").strip().lower()
        if decision not in ("son lo mismo", "no es lo mismo"):
            decision = ""

        w.writerow({
            "row_id": row_id,
            "opportunity_id": norm_text(r.get("opportunity_id")),
            "opportunity_name": norm_text(r.get("opportunity_name")),
            "source_actor_name": norm_text(r.get("source_actor_name")),
            "source_actor_type": norm_lower(r.get("source_actor_type")),
            "instrument_type": norm_lower(r.get("instrument_type")),
            "geographic_scope": norm_lower(r.get("geographic_scope")),
            "sector_scope": norm_lower(r.get("sector_scope")),
            "status": norm_lower(r.get("status")),
            "application_window": norm_text(r.get("application_window")),
            "source_url": norm_text(r.get("source_url")),
            "provider_actor_id": norm_text(r.get("provider_actor_id")),
            "notes_internal": norm_text(r.get("notes_internal")),
            "last_updated_at": norm_text(r.get("last_updated")),
            "doc_source": norm_text(r.get("Doc_Source")),
            "duplicate_group_id": (r.get("Grupo_Duplicado") or "").strip(),
            "duplicate_decision": decision,
            "eligible_actor_types_array": json.dumps(eligible, ensure_ascii=False),
            "gpc_sector_array": json.dumps(gpc_sector, ensure_ascii=False),
            "gpc_subsector_array": json.dumps(gpc_subsector, ensure_ascii=False),
            "gpc_sector_raw": gpc_raw,
        })

print("Wrote:", FUNDING_NORMALIZED_OUT.name)

Wrote: funding_database_v1_normalized.csv


In [12]:
# 2) Compute calibrated action fundability score

with ACTION_INPUT.open("r", encoding="utf-8-sig", newline="") as f:
    actions = list(csv.DictReader(f))

SECTOR_MATCH = {
    "stationary_energy": ["energy", "stationary energy", "building", "buildings"],
    "transportation": ["transport", "transportation", "mobility", "fleet"],
    "waste": ["waste", "landfill", "recycling", "residual", "organic"],
    "ippu_industry": ["industry", "industrial", "ippu", "manufacturing"],
    "afolu": ["afolu", "agriculture", "forest", "land use", "agro", "peatland"],
}

CAPEX_TYPES = {"infrastructure"}
POLICY_TYPES = {"regulatory", "planning"}
PROGRAM_TYPES = {"program", "financial"}


def sector_fit(action_sector, fund_row):
    s = (fund_row.get("sector_scope") or "").lower()
    g = (fund_row.get("gpc_sector") or "").lower()
    keys = SECTOR_MATCH.get(action_sector, [])
    direct = any(k in s for k in keys) or any(k in g for k in keys)
    cross = "cross_sector" in s
    return direct, cross


def actor_fit_score(agent, fund_row):
    e = (fund_row.get("eligible_actor_types") or "").lower()
    if agent == "municipal_government":
        if "municipal" in e:
            return 2
        if "public_agency" in e or "regional" in e or "national" in e:
            return 1
        return 0
    if agent == "utility_or_energy_operator":
        if "public_agency" in e or "private" in e:
            return 2
        return 1 if e else 0
    if agent == "private_sector_industry":
        if "private" in e:
            return 2
        if "public_agency" in e:
            return 1
        return 0
    if agent == "land_sector_actors":
        if "community_org" in e or "ngo" in e or "private" in e or "municipal" in e:
            return 2
        return 1 if e else 0
    return 1 if e else 0


def geography_fit_score(fund_row):
    g = (fund_row.get("geographic_scope") or "").lower()
    if "chile comunal" in g or "chile regional" in g:
        return 2
    if "chile national" in g:
        return 1
    if "latin america" in g:
        return 1
    return 0


def instrument_fit_score(action_row, fund_row):
    itype = (action_row.get("intervention_type") or "").strip().lower()
    inst = (fund_row.get("instrument_type") or "").lower()
    if itype in CAPEX_TYPES:
        return 2 if inst in ("grant", "loan", "concessional_loan", "blended") else 0
    if itype in POLICY_TYPES:
        return 2 if inst in ("technical_assistance", "grant") else 0
    if itype in PROGRAM_TYPES:
        return 2 if inst in ("grant", "technical_assistance", "blended", "loan") else 1
    return 1 if inst in ("grant", "technical_assistance", "loan", "blended") else 0


def timing_fit_score(fund_row):
    st = (fund_row.get("status") or "").lower()
    if st in ("open", "recurring"):
        return 2
    if st in ("other", "no especificado", ""):
        return 1
    return 0


def constraint_fit_score(fund_row):
    notes = (fund_row.get("notes_internal") or "").lower()
    barrier_terms = ["falta de capacidad", "alto costo", "riesgo", "mercado inmaduro", "barrera"]
    return 0 if any(t in notes for t in barrier_terms) else 1


def specificity_score(fund_row):
    s = (fund_row.get("sector_scope") or "").lower()
    g = (fund_row.get("gpc_sector") or "")
    token_count = len([x for x in g.split(",") if x.strip()])
    if "cross_sector" in s:
        return 0.2
    if token_count >= 4:
        return 0.5
    if token_count >= 2:
        return 0.8
    return 1.0


def to100(raw11):
    return round((raw11 / 11) * 100, 2)


def damp(x):
    # diminishing returns in [0,100]
    return 100 * (1 - (2.718281828 ** (-(x / 35))))

rows_out = []
for a in actions:
    direct_matches = []
    cross_matches = []

    for f in funding_rows:
        direct, cross = sector_fit(a.get("sector", ""), f)
        if not direct and not cross:
            continue

        actor = actor_fit_score(a.get("implementing_agent", ""), f)
        geo = geography_fit_score(f)

        # minimum viability gate
        if actor + geo < 2:
            continue

        inst = instrument_fit_score(a, f)
        timing = timing_fit_score(f)
        constraints = constraint_fit_score(f)

        if direct:
            raw = 2 + actor + geo + inst + timing + constraints  # max 11
            direct_matches.append((raw, f.get("opportunity_id", ""), specificity_score(f)))
        else:
            # cross-sector can only contribute as a small bonus later
            cross_matches.append((f.get("opportunity_id", ""), specificity_score(f)))

    direct_matches.sort(key=lambda x: x[0], reverse=True)

    if not direct_matches:
        rows_out.append({
            "action_id": a.get("action_id", ""),
            "action_name": a.get("action_name", ""),
            "sector": a.get("sector", ""),
            "subsector": a.get("subsector", ""),
            "implementing_agent": a.get("implementing_agent", ""),
            "intervention_type": a.get("intervention_type", ""),
            "matched_opportunity_count": 0,
            "avg_match_score_100": "0.00",
            "top3_avg_score_100": "0.00",
            "high_fit_count": 0,
            "fundability_score_100": "0.00",
            "fundability_band": "very_low",
            "top_opportunity_ids": "",
        })
        continue

    raw_vals = [m[0] for m in direct_matches]
    top3_vals = raw_vals[:3]
    top5_vals = raw_vals[:5]

    avg_score = to100(sum(raw_vals) / len(raw_vals))
    top3_avg = to100(sum(top3_vals) / len(top3_vals))
    high_fit = sum(1 for r in raw_vals if r >= 8)

    # Depth: count only specific direct opportunities, with diminishing returns
    weighted_depth = sum(spec for _, _, spec in direct_matches)
    depth_component = min(100, damp(weighted_depth))

    # Quality: top5 mean, then damp to avoid ceiling effects
    quality_component = damp(to100(sum(top5_vals) / len(top5_vals)))

    # High-fit share: stricter and bounded
    high_share_component = (high_fit / len(direct_matches)) * 100

    # Very small cross-sector bonus
    cross_bonus = min(3.0, sum(spec for _, spec in cross_matches) * 0.02)

    # Penalize repetitive generic top options
    top_ids = [oid for _, oid, _ in direct_matches[:5] if oid]
    unique_top_ratio = len(set(top_ids)) / max(1, len(top_ids))
    generic_penalty = 12 if unique_top_ratio < 0.6 else (6 if unique_top_ratio < 0.8 else 0)

    fundability = round(
        0.25 * depth_component
        + 0.55 * quality_component
        + 0.20 * high_share_component
        + cross_bonus
        - generic_penalty,
        2,
    )
    fundability = max(0.0, min(100.0, fundability))

    # Absolute interpretation (fixed thresholds)
    if fundability >= 80:
        abs_band = "high"
    elif fundability >= 65:
        abs_band = "medium"
    elif fundability >= 50:
        abs_band = "low"
    else:
        abs_band = "very_low"

    rows_out.append({
        "action_id": a.get("action_id", ""),
        "action_name": a.get("action_name", ""),
        "sector": a.get("sector", ""),
        "subsector": a.get("subsector", ""),
        "implementing_agent": a.get("implementing_agent", ""),
        "intervention_type": a.get("intervention_type", ""),
        "matched_opportunity_count": len(direct_matches),
        "avg_match_score_100": f"{avg_score:.2f}",
        "top3_avg_score_100": f"{top3_avg:.2f}",
        "high_fit_count": high_fit,
        "fundability_score_100": f"{fundability:.2f}",
        "fundability_band_relative": "pending",
        "fundability_band_absolute": abs_band,
        "fundability_band": "pending",
        "top_opportunity_ids": "|".join(top_ids),
    })

# Percentile-based bands to guarantee spread
rows_out.sort(key=lambda r: float(r["fundability_score_100"]), reverse=True)
N = len(rows_out)
for i, r in enumerate(rows_out):
    pct = (i + 1) / N
    if pct <= 0.20:
        band = "high"
    elif pct <= 0.55:
        band = "medium"
    elif pct <= 0.85:
        band = "low"
    else:
        band = "very_low"
    r["fundability_band_relative"] = band
    r["fundability_band"] = band

with ACTION_FUNDABILITY_OUT.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows_out[0].keys()))
    w.writeheader()
    w.writerows(rows_out)

print("Wrote:", ACTION_FUNDABILITY_OUT.name)

Wrote: action_fundability_scores_v1.csv
